# cuecard — Step-by-Step Pipeline Visualization

Interactive exploration of each pipeline step: parse, embed, index, retrieve, format.
Plus evaluation metrics and model comparison.

**D17 Rule:** Zero pipeline logic in this notebook. All computation delegates to `cuecard` imports.

In [ ]:
import sys
from pathlib import Path

# Ensure cuecard is importable
project_root = Path.cwd().parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import numpy as np
import matplotlib.pyplot as plt

print(f"Project root: {project_root}")

## Step 1: Parse Rules

Load and display all rules from a corpus file with provenance.

In [ ]:
from cuecard.parser import parse_rules

CORPUS_PATH = str(project_root / "eval" / "corpora" / "rules_basic.txt")
rules = parse_rules((CORPUS_PATH,))

print(f"Parsed {len(rules)} rules from {CORPUS_PATH}\n")
for i, rule in enumerate(rules, 1):
    print(f"  {i:3d}  {rule.text}")
    print(f"       {rule.provenance.file}:{rule.provenance.line_start}")

### Rule Length Distribution

In [ ]:
lengths = [len(r.text) for r in rules]
word_counts = [len(r.text.split()) for r in rules]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(lengths, bins=20, edgecolor="black", alpha=0.7)
ax1.set_xlabel("Character count")
ax1.set_ylabel("Number of rules")
ax1.set_title("Rule Length Distribution (chars)")
ax1.axvline(np.mean(lengths), color="red", linestyle="--", label=f"mean={np.mean(lengths):.0f}")
ax1.legend()

ax2.hist(word_counts, bins=15, edgecolor="black", alpha=0.7, color="orange")
ax2.set_xlabel("Word count")
ax2.set_ylabel("Number of rules")
ax2.set_title("Rule Length Distribution (words)")
ax2.axvline(np.mean(word_counts), color="red", linestyle="--", label=f"mean={np.mean(word_counts):.1f}")
ax2.legend()

plt.tight_layout()
plt.show()

## Step 2: Embed Rules

Encode rules using the embedding model (asymmetric: `passage_embed()`).

In [ ]:
from fastembed import TextEmbedding
from cuecard.indexer import build_index
from cuecard.freshness import check_freshness

MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = TextEmbedding(model_name=MODEL_NAME)

freshness = check_freshness((CORPUS_PATH,), {})
index = build_index(
    tuple(rules),
    freshness.updated_sources,
    MODEL_NAME,
    model=model,
)

print(f"Index: {index}")
print(f"Embeddings shape: {index.embeddings.shape}")
print(f"L2 norms (should be ~1.0): {np.linalg.norm(index.embeddings, axis=1)[:5]}")

### Embedding Similarity Heatmap

All-pairs cosine similarity between rules.

In [ ]:
sim_matrix = index.embeddings @ index.embeddings.T

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="RdYlBu_r", vmin=0, vmax=1)
ax.set_title("Rule-Rule Cosine Similarity")
ax.set_xlabel("Rule index")
ax.set_ylabel("Rule index")
plt.colorbar(im, ax=ax, shrink=0.8)

# Annotate with short rule labels
short_labels = [r.text[:30] + "..." if len(r.text) > 30 else r.text for r in rules]
ax.set_xticks(range(len(rules)))
ax.set_yticks(range(len(rules)))
ax.set_xticklabels(range(1, len(rules) + 1), fontsize=7)
ax.set_yticklabels([f"{i+1}: {l}" for i, l in enumerate(short_labels)], fontsize=6)

plt.tight_layout()
plt.show()

### t-SNE Visualization of Rule Embeddings

In [ ]:
from sklearn.manifold import TSNE

perplexity = min(5, len(rules) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
embeddings_2d = tsne.fit_transform(index.embeddings)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=80, alpha=0.7)

for i, rule in enumerate(rules):
    label = rule.text[:40] + "..." if len(rule.text) > 40 else rule.text
    ax.annotate(label, (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                fontsize=6, alpha=0.8, ha="left", va="bottom")

ax.set_title(f"t-SNE of Rule Embeddings ({MODEL_NAME})")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()

## Step 3: Retrieve

Run queries against the index (asymmetric: `query_embed()`).

In [ ]:
from cuecard.retriever import retrieve
from cuecard.formatter import format_rules, format_rules_verbose

queries = [
    "Bash: git commit -m 'fix auth bug'",
    "Bash: tmux send-keys -t cody 'hello'",
    "Bash: pip install requests",
    "Edit: src/auth.py: def login(user_input)",
    "Bash: python -c 'eval(input())'",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    results = retrieve(index, query, model=model, threshold=0.2, top_k=5)
    print(format_rules_verbose(results))

### Score Distribution Across All Fixtures

In [ ]:
import json

FIXTURES_PATH = str(project_root / "eval" / "fixtures" / "basic.json")
with open(FIXTURES_PATH) as f:
    fixtures_raw = json.load(f)

all_scores = []
match_scores = []
non_match_scores = []

for fixture in fixtures_raw:
    results = retrieve(index, fixture["query"], model=model, threshold=0.0, top_k=50)
    should_match = set(fixture["should_match"])
    should_not_match = set(fixture["should_not_match"])
    
    for r in results:
        all_scores.append(r.score)
        if r.rule.text in should_match:
            match_scores.append(r.score)
        elif r.rule.text in should_not_match:
            non_match_scores.append(r.score)

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 1, 40)
ax.hist(all_scores, bins=bins, alpha=0.4, label=f"All ({len(all_scores)})", color="gray")
ax.hist(match_scores, bins=bins, alpha=0.7, label=f"should_match ({len(match_scores)})", color="green")
ax.hist(non_match_scores, bins=bins, alpha=0.7, label=f"should_not_match ({len(non_match_scores)})", color="red")

# Default threshold
ax.axvline(0.35, color="blue", linestyle="--", linewidth=2, label="threshold=0.35")

ax.set_xlabel("Cosine Similarity Score")
ax.set_ylabel("Count")
ax.set_title("Score Distribution: Relevant vs Irrelevant Rules")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nShould-match scores: mean={np.mean(match_scores):.3f}, min={np.min(match_scores):.3f}, max={np.max(match_scores):.3f}")
if non_match_scores:
    print(f"Should-not-match scores: mean={np.mean(non_match_scores):.3f}, min={np.min(non_match_scores):.3f}, max={np.max(non_match_scores):.3f}")
else:
    print("Should-not-match: none retrieved (good!)")

## Step 4: Format

Show the final injectable text that gets injected into agent context.

In [ ]:
sample_query = "Bash: git commit -m 'fix auth bug'"
results = retrieve(index, sample_query, model=model, top_k=5)

print("Formatted output (injected into additionalContext):")
print("---")
print(format_rules(results))
print("---")

## Step 5: Evaluation

Run the full evaluation framework on golden fixtures.

In [ ]:
from cuecard.eval import load_fixtures, run_eval, format_eval_report

fixtures = load_fixtures(FIXTURES_PATH)
corpus_dir = str(project_root / "eval" / "corpora")

summary = run_eval(
    fixtures,
    corpus_dir=corpus_dir,
    model_name=MODEL_NAME,
    model=model,
    top_k=5,
    threshold=0.35,
)

print(format_eval_report(summary))

### Per-Fixture Metrics Visualization

In [ ]:
fixture_ids = [r.fixture_id for r in summary.per_fixture]
precisions = [r.precision_at_k for r in summary.per_fixture]
recalls = [r.recall_at_k for r in summary.per_fixture]
noise = [r.noise_ratio for r in summary.per_fixture]
waste = [r.context_waste_ratio for r in summary.per_fixture]
difficulties = [r.difficulty for r in summary.per_fixture]

# Color-code by difficulty
diff_colors = {"easy": "#4CAF50", "medium": "#FF9800", "hard": "#F44336", "negative": "#9E9E9E"}
colors = [diff_colors.get(d, "#2196F3") for d in difficulties]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Top: Recall vs Precision per fixture
x = np.arange(len(fixture_ids))
width = 0.25
ax1.bar(x - width, precisions, width, label="Precision@k", alpha=0.8, color="blue")
ax1.bar(x, recalls, width, label="Recall@k", alpha=0.8, color="green")
ax1.bar(x + width, noise, width, label="Noise Ratio", alpha=0.8, color="red")
ax1.set_ylabel("Score")
ax1.set_title(f"Per-Fixture: Precision vs Recall vs Noise ({MODEL_NAME})")
ax1.set_xticks(x)
ax1.set_xticklabels(fixture_ids, rotation=45, ha="right", fontsize=6)
ax1.legend()
ax1.set_ylim(0, 1.1)

# Bottom: Context waste per fixture (colored by difficulty)
ax2.bar(x, waste, color=colors, alpha=0.8)
ax2.set_ylabel("Context Waste Ratio")
ax2.set_title("Per-Fixture Context Waste (colored by difficulty)")
ax2.set_xticks(x)
ax2.set_xticklabels(fixture_ids, rotation=45, ha="right", fontsize=6)
ax2.set_ylim(0, 1.1)
# Legend for difficulty colors
for tier, color in diff_colors.items():
    ax2.bar([], [], color=color, label=tier)
ax2.legend()

plt.tight_layout()
plt.show()

### Latency Distribution

In [ ]:
latencies = [r.latency_ms for r in summary.per_fixture]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(fixture_ids, latencies, alpha=0.7, color="steelblue")
ax.axhline(summary.latency_p50_ms, color="green", linestyle="--", label=f"p50={summary.latency_p50_ms:.1f}ms")
ax.axhline(summary.latency_p95_ms, color="orange", linestyle="--", label=f"p95={summary.latency_p95_ms:.1f}ms")
ax.axhline(summary.latency_p99_ms, color="red", linestyle="--", label=f"p99={summary.latency_p99_ms:.1f}ms")

ax.set_xlabel("Fixture")
ax.set_ylabel("Latency (ms)")
ax.set_title("Retrieval Latency per Fixture")
ax.set_xticklabels(fixture_ids, rotation=45, ha="right", fontsize=7)
ax.legend()
plt.tight_layout()
plt.show()

### Aggregate Summary

In [ ]:
print(f"{'Metric':<25} {'Value':>10}")
print(f"{'-'*25} {'-'*10}")
print(f"{'Fixtures':<25} {summary.fixture_count:>10d}")
print(f"{'Mean Precision@k':<25} {summary.mean_precision:>10.3f}")
print(f"{'Mean Recall@k':<25} {summary.mean_recall:>10.3f}")
print(f"{'Mean MRR':<25} {summary.mean_mrr:>10.3f}")
print(f"{'Mean nDCG@k':<25} {summary.mean_ndcg:>10.3f}")
print(f"{'Mean Anti-Precision':<25} {summary.mean_anti_precision:>10.3f}")
print(f"{'Mean Noise Ratio':<25} {summary.mean_noise_ratio:>10.3f}")
print(f"{'Mean Context Waste':<25} {summary.mean_context_waste_ratio:>10.3f}")
print(f"{'Neg Silence Rate':<25} {summary.negative_silence_rate:>10.3f}")
print(f"{'Mean Retrieved Count':<25} {summary.mean_retrieved_count:>10.1f}")
print(f"{'Latency p50 (ms)':<25} {summary.latency_p50_ms:>10.1f}")
print(f"{'Latency p95 (ms)':<25} {summary.latency_p95_ms:>10.1f}")
print(f"{'Latency p99 (ms)':<25} {summary.latency_p99_ms:>10.1f}")

### Per-Tier Breakdown

How do metrics differ across difficulty tiers? Negative fixtures are the precision stress test.

In [ ]:
# Configure models to compare (uncomment those you've downloaded)
MODELS_TO_COMPARE = [
    "BAAI/bge-small-en-v1.5",
    # "nomic-ai/nomic-embed-text-v1.5",
    # "snowflake/snowflake-arctic-embed-m",
    # "jinaai/jina-embeddings-v2-base-code",
    # "mixedbread-ai/mxbai-embed-large-v1",
]

comparison_results = {}
for model_name in MODELS_TO_COMPARE:
    print(f"\nEvaluating: {model_name}")
    m = TextEmbedding(model_name=model_name)
    s = run_eval(fixtures, corpus_dir=corpus_dir, model_name=model_name, model=m)
    comparison_results[model_name] = s
    print(f"  P@k={s.mean_precision:.3f}  R@k={s.mean_recall:.3f}  "
          f"MRR={s.mean_mrr:.3f}  Noise={s.mean_noise_ratio:.3f}  "
          f"Waste={s.mean_context_waste_ratio:.3f}  Silence={s.negative_silence_rate:.3f}  "
          f"AvgRet={s.mean_retrieved_count:.1f}  p50={s.latency_p50_ms:.1f}ms")

## Step 6: Model Comparison

Compare different embedding models on the same fixtures.

**Note:** Each model must be downloaded first via `cuecard setup --model <name>`.

In [ ]:
# Configure models to compare (uncomment those you've downloaded)
MODELS_TO_COMPARE = [
    "BAAI/bge-small-en-v1.5",
    # "nomic-ai/nomic-embed-text-v1.5",
    # "snowflake/snowflake-arctic-embed-m",
    # "jinaai/jina-embeddings-v2-base-code",
    # "mixedbread-ai/mxbai-embed-large-v1",
]

comparison_results = {}
for model_name in MODELS_TO_COMPARE:
    print(f"\nEvaluating: {model_name}")
    m = TextEmbedding(model_name=model_name)
    s = run_eval(fixtures, corpus_dir=corpus_dir, model_name=model_name, model=m)
    comparison_results[model_name] = s
    print(f"  Precision={s.mean_precision:.3f}  Recall={s.mean_recall:.3f}  "
          f"MRR={s.mean_mrr:.3f}  nDCG={s.mean_ndcg:.3f}  "
          f"AntiPrec={s.mean_anti_precision:.3f}  p50={s.latency_p50_ms:.1f}ms")

In [ ]:
thresholds = np.arange(0.0, 0.8, 0.05)
threshold_precisions = []
threshold_recalls = []
threshold_noise = []
threshold_waste = []
threshold_silence = []
threshold_avg_ret = []

for t in thresholds:
    s = run_eval(fixtures, corpus_dir=corpus_dir, model_name=MODEL_NAME,
                 model=model, threshold=float(t))
    threshold_precisions.append(s.mean_precision)
    threshold_recalls.append(s.mean_recall)
    threshold_noise.append(s.mean_noise_ratio)
    threshold_waste.append(s.mean_context_waste_ratio)
    threshold_silence.append(s.negative_silence_rate)
    threshold_avg_ret.append(s.mean_retrieved_count)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Recall vs Precision vs Noise
ax1.plot(thresholds, threshold_recalls, "s-", label="Recall@k", linewidth=2, color="green")
ax1.plot(thresholds, threshold_precisions, "o-", label="Precision@k", linewidth=2, color="blue")
ax1.plot(thresholds, threshold_noise, "^-", label="Noise Ratio", linewidth=2, color="red")
ax1.plot(thresholds, threshold_waste, "D-", label="Context Waste", linewidth=2, color="orange")
ax1.axvline(0.30, color="gray", linestyle="--", alpha=0.5, label="default=0.30")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Score")
ax1.set_title("Threshold vs Quality Metrics")
ax1.legend(fontsize=8)
ax1.set_ylim(-0.05, 1.1)

# Right: Silence rate + avg retrieved count
ax2.plot(thresholds, threshold_silence, "s-", label="Neg Silence Rate", linewidth=2, color="purple")
ax2_twin = ax2.twinx()
ax2_twin.plot(thresholds, threshold_avg_ret, "o-", label="Avg Retrieved", linewidth=2, color="steelblue")
ax2.set_xlabel("Threshold")
ax2.set_ylabel("Silence Rate")
ax2_twin.set_ylabel("Avg Retrieved Count")
ax2.set_title("Threshold vs Context Usage")
ax2.legend(loc="center left")
ax2_twin.legend(loc="center right")
ax2.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.show()

## Step 7: Threshold Sensitivity

How do metrics change across different threshold values?

In [ ]:
thresholds = np.arange(0.0, 0.8, 0.05)
threshold_precisions = []
threshold_recalls = []
threshold_anti = []

for t in thresholds:
    s = run_eval(fixtures, corpus_dir=corpus_dir, model_name=MODEL_NAME,
                 model=model, threshold=float(t))
    threshold_precisions.append(s.mean_precision)
    threshold_recalls.append(s.mean_recall)
    threshold_anti.append(s.mean_anti_precision)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, threshold_precisions, "o-", label="Precision@k", linewidth=2)
ax.plot(thresholds, threshold_recalls, "s-", label="Recall@k", linewidth=2)
ax.plot(thresholds, threshold_anti, "^-", label="Anti-Precision", linewidth=2, color="red")
ax.axvline(0.35, color="gray", linestyle="--", alpha=0.5, label="default=0.35")

ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold Sensitivity Analysis")
ax.legend()
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()

## Step 8: Workflow Rules & UserPromptSubmit Evaluation

Evaluate workflow/process rules retrieved for user messages (not tool calls).
This tests the unified event system — can the pipeline distinguish coding rules from workflow rules?

In [ ]:
# Load workflow corpus and fixtures
WORKFLOW_CORPUS = str(project_root / "eval" / "corpora" / "rules_workflow.txt")
WORKFLOW_FIXTURES = str(project_root / "eval" / "fixtures" / "workflow.json")

workflow_rules = parse_rules((WORKFLOW_CORPUS,))
print(f"Workflow rules: {len(workflow_rules)}")
for i, rule in enumerate(workflow_rules, 1):
    print(f"  {i:3d}  {rule.text[:80]}{'...' if len(rule.text) > 80 else ''}")

workflow_fixtures = load_fixtures(WORKFLOW_FIXTURES)
from collections import Counter
tiers = Counter(f.difficulty for f in workflow_fixtures)
print(f"\nWorkflow fixtures: {len(workflow_fixtures)}")
for tier, count in sorted(tiers.items()):
    print(f"  {tier}: {count}")

In [ ]:
# Evaluate workflow fixtures (embedding-only)
workflow_summary = run_eval(
    workflow_fixtures,
    corpus_dir=corpus_dir,
    model_name=MODEL_NAME,
    model=model,
    top_k=5,
    threshold=0.30,
)
print("=== Workflow Evaluation (embedding-only) ===")
print(format_eval_report(workflow_summary))

### Step 9: Unified Index Evaluation

Test both PreToolUse and UserPromptSubmit fixtures against a single unified index
containing both coding rules AND workflow rules. This is the cross-domain noise test.

In [ ]:
# Load combined fixtures and run against unified corpus
COMBINED_FIXTURES = str(project_root / "eval" / "fixtures" / "combined.json")
combined_fixtures = load_fixtures(COMBINED_FIXTURES)
print(f"Combined fixtures: {len(combined_fixtures)}")

# Build unified index from both corpora
unified_corpus = (CORPUS_PATH, WORKFLOW_CORPUS)
unified_summary = run_eval(
    combined_fixtures,
    corpus_dir=corpus_dir,
    model_name=MODEL_NAME,
    model=model,
    top_k=5,
    threshold=0.30,
    corpus_override=unified_corpus,
)

print("=== Unified Index Evaluation (embedding-only, both corpora) ===")
print(format_eval_report(unified_summary))

# Compare: separate vs unified
print("\n=== Comparison: Separate vs Unified ===")
print(f"{'Metric':<20} {'Basic':>10} {'Workflow':>10} {'Unified':>10}")
print(f"{'-'*20} {'-'*10} {'-'*10} {'-'*10}")
print(f"{'Recall':<20} {summary.mean_recall:>10.3f} {workflow_summary.mean_recall:>10.3f} {unified_summary.mean_recall:>10.3f}")
print(f"{'Noise':<20} {summary.mean_noise_ratio:>10.3f} {workflow_summary.mean_noise_ratio:>10.3f} {unified_summary.mean_noise_ratio:>10.3f}")
print(f"{'Neg Silence':<20} {summary.negative_silence_rate:>10.3f} {workflow_summary.negative_silence_rate:>10.3f} {unified_summary.negative_silence_rate:>10.3f}")
print(f"{'Precision':<20} {summary.mean_precision:>10.3f} {workflow_summary.mean_precision:>10.3f} {unified_summary.mean_precision:>10.3f}")

## Step 10: Enriched Index Visualization

Explore the enriched rules.json — expansions per rule, total embedding rows, and sample expansions.

In [ ]:
import json

ENRICHED_RULES_PATH = str(project_root / "eval" / "corpora" / "enriched_basic_v3" / "rules.json")
with open(ENRICHED_RULES_PATH) as f:
    enriched_data = json.load(f)

enriched_rules = enriched_data["rules"]
exp_counts = [len(r.get("expansions", [])) for r in enriched_rules]
total_embeddings = len(enriched_rules) + sum(exp_counts)  # 1 canonical + N expansions each

print(f"Rules: {len(enriched_rules)}")
print(f"Total expansions: {sum(exp_counts)}")
print(f"Total embedding rows: {total_embeddings} ({len(enriched_rules)} canonical + {sum(exp_counts)} expansions)")
print(f"Expansions per rule: min={min(exp_counts)}, max={max(exp_counts)}, mean={np.mean(exp_counts):.1f}")

# Sample expansions for 3 rules
for i in [0, 10, 20]:
    r = enriched_rules[i]
    exps = r.get("expansions", [])
    print(f"\n{'='*60}")
    print(f"Rule {i}: {r['text']}")
    print(f"Expansions ({len(exps)}):")
    for e in exps[:4]:
        print(f"  - {e}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of expansions per rule
ax1.bar(range(len(exp_counts)), sorted(exp_counts, reverse=True),
        edgecolor="black", alpha=0.7, color="steelblue")
ax1.axhline(np.mean(exp_counts), color="red", linestyle="--",
            label=f"mean={np.mean(exp_counts):.1f}")
ax1.set_xlabel("Rule (sorted by expansion count)")
ax1.set_ylabel("Number of expansions")
ax1.set_title("Expansions per Rule (v3 prompt)")
ax1.legend()

# Pie: canonical vs expansion embedding rows
ax2.pie([len(enriched_rules), sum(exp_counts)],
        labels=[f"Canonical ({len(enriched_rules)})",
                f"Expansions ({sum(exp_counts)})"],
        autopct="%1.0f%%", colors=["#4CAF50", "#FF9800"],
        startangle=90)
ax2.set_title(f"Embedding Row Composition ({total_embeddings} total)")

plt.tight_layout()
plt.show()

## Step 11: Raw vs Enriched Comparison

Compare raw embeddings (no expansions) against enriched embeddings (v3 expansions + BM25 + RRF).
Both use bge-small, embedding mode only (no LLM reranker).

In [ ]:
RESULTS_DIR = project_root / "eval" / "results"

with open(RESULTS_DIR / "raw-baseline-2026-04-02.json") as f:
    raw_results = json.load(f)
with open(RESULTS_DIR / "enriched-embedding-2026-04-02.json") as f:
    enriched_results = json.load(f)

# Bar chart: recall, noise, precision for raw vs enriched across fixture sets
fixture_sets = ["basic", "workflow"]
metrics = ["recall", "noise", "precision"]

# Normalize key names (raw uses short keys, enriched uses mean_* keys)
def get_metric(d, metric):
    """Extract metric from either raw or enriched result format."""
    key_map = {"recall": ("recall", "mean_recall"),
               "noise": ("noise", "mean_noise_ratio"),
               "precision": ("precision", "mean_precision")}
    for k in key_map[metric]:
        if k in d:
            return d[k]
    return 0.0

fig, axes = plt.subplots(1, len(fixture_sets), figsize=(14, 5), sharey=True)
colors_raw = {"recall": "#2196F3", "noise": "#F44336", "precision": "#4CAF50"}
colors_enr = {"recall": "#90CAF9", "noise": "#EF9A9A", "precision": "#A5D6A7"}

for idx, fset in enumerate(fixture_sets):
    ax = axes[idx]
    x = np.arange(len(metrics))
    width = 0.35

    raw_vals = [get_metric(raw_results[fset], m) for m in metrics]
    enr_vals = [get_metric(enriched_results[fset], m) for m in metrics]

    bars1 = ax.bar(x - width / 2, raw_vals, width, label="Raw",
                   color=[colors_raw[m] for m in metrics], edgecolor="black", alpha=0.8)
    bars2 = ax.bar(x + width / 2, enr_vals, width, label="Enriched",
                   color=[colors_enr[m] for m in metrics], edgecolor="black", alpha=0.8)

    # Annotate deltas
    for i, (r, e) in enumerate(zip(raw_vals, enr_vals)):
        delta = e - r
        sign = "+" if delta >= 0 else ""
        ax.annotate(f"{sign}{delta:.1%}", xy=(x[i] + width / 2, e),
                    ha="center", va="bottom", fontsize=8, fontweight="bold",
                    color="green" if (delta > 0 and metrics[i] != "noise") or
                          (delta < 0 and metrics[i] == "noise") else "red")

    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_title(f"{fset.capitalize()} ({raw_results[fset].get('fixtures', enriched_results[fset].get('fixture_count', '?'))} fixtures)")
    ax.set_ylim(0, 1.1)
    ax.legend()

fig.suptitle("Raw vs Enriched (bge-small, embedding mode)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Per-tier comparison table: Raw vs Enriched (basic fixtures)
print("Per-Tier Comparison: Raw vs Enriched (basic, bge-small, embedding mode)")
print(f"{'Tier':<10} {'Raw Recall':>12} {'Enr Recall':>12} {'Delta':>8} {'Raw Noise':>12} {'Enr Noise':>12} {'Delta':>8}")
print(f"{'-'*10} {'-'*12} {'-'*12} {'-'*8} {'-'*12} {'-'*12} {'-'*8}")

# Raw baseline doesn't have per_tier, but enriched does
enriched_tiers = {t["tier"]: t for t in enriched_results["basic"]["per_tier"]}

# Use model-comparison bge-small for raw per-tier (same data, has per_tier)
with open(RESULTS_DIR / "model-comparison-2026-04-02.json") as f:
    model_comp = json.load(f)
raw_bge = model_comp["BAAI/bge-small-en-v1.5"]
raw_tiers = {t["tier"]: t for t in raw_bge["basic"]["per_tier"]}

for tier in ["easy", "medium", "hard", "negative"]:
    rt = raw_tiers.get(tier, {})
    et = enriched_tiers.get(tier, {})
    raw_r = rt.get("mean_recall", 0)
    enr_r = et.get("mean_recall", 0)
    raw_n = rt.get("mean_noise_ratio", 0)
    enr_n = et.get("mean_noise_ratio", 0)
    dr = enr_r - raw_r
    dn = enr_n - raw_n
    print(f"{tier:<10} {raw_r:>12.1%} {enr_r:>12.1%} {dr:>+8.1%} {raw_n:>12.1%} {enr_n:>12.1%} {dn:>+8.1%}")

## Step 12: Model Comparison (Enriched)

Compare 6 embedding models on enriched corpora. All use the same enriched index (v3 expansions + BM25 + RRF), embedding mode only.

In [ ]:
# Model comparison bar chart: recall and noise for basic and workflow
model_names_short = {
    "jinaai/jina-embeddings-v2-base-code": "jina-code-v2",
    "mixedbread-ai/mxbai-embed-large-v1": "mxbai-large",
    "nomic-ai/nomic-embed-text-v1.5": "nomic-v1.5",
    "snowflake/snowflake-arctic-embed-m": "arctic-m",
    "BAAI/bge-small-en-v1.5": "bge-small",
    "jinaai/jina-embeddings-v3": "jina-v3",
}

# Sort by basic recall descending
model_order = sorted(model_comp.keys(),
                     key=lambda m: model_comp[m]["basic"]["mean_recall"],
                     reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for idx, fset in enumerate(["basic", "workflow"]):
    ax = axes[idx]
    x = np.arange(len(model_order))
    width = 0.35

    recalls = [model_comp[m][fset]["mean_recall"] for m in model_order]
    noises = [model_comp[m][fset]["mean_noise_ratio"] for m in model_order]
    labels = [model_names_short.get(m, m.split("/")[-1]) for m in model_order]

    ax.bar(x - width / 2, recalls, width, label="Recall", color="#4CAF50", alpha=0.8)
    ax.bar(x + width / 2, noises, width, label="Noise", color="#F44336", alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{fset.capitalize()} Fixtures")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.1)
    ax.legend()

fig.suptitle("Embedding Model Comparison (enriched, embedding mode)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Per-tier heatmap: recall by model and tier (basic fixtures)
tiers = ["easy", "medium", "hard"]
labels = [model_names_short.get(m, m.split("/")[-1]) for m in model_order]

recall_matrix = np.zeros((len(model_order), len(tiers)))
for i, m in enumerate(model_order):
    tier_data = {t["tier"]: t for t in model_comp[m]["basic"]["per_tier"]}
    for j, tier in enumerate(tiers):
        recall_matrix[i, j] = tier_data.get(tier, {}).get("mean_recall", 0)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(recall_matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(tiers)))
ax.set_xticklabels([t.capitalize() for t in tiers])
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_title("Recall by Model and Tier (basic, enriched embedding)")

# Annotate cells
for i in range(len(labels)):
    for j in range(len(tiers)):
        ax.text(j, i, f"{recall_matrix[i, j]:.1%}",
                ha="center", va="center", fontsize=10,
                color="white" if recall_matrix[i, j] < 0.5 else "black")

plt.colorbar(im, ax=ax, shrink=0.8, label="Recall")
plt.tight_layout()
plt.show()

## Step 13: LLM Reranker Impact

Before/after comparison: enriched embedding-only vs enriched + LLM reranker (Qwen3.5-35B, reasoning prompt).
The LLM reranker is the single biggest quality lever — it collapses noise from ~84% to ~21%.

In [ ]:
with open(RESULTS_DIR / "enriched-llm-local-2026-04-02.json") as f:
    llm_results = json.load(f)

# Before/after chart: embedding-only vs embedding+LLM
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_metrics = ["noise", "neg_silence", "recall"]
bar_labels = ["Noise Ratio", "Neg Silence", "Recall"]
colors_before = ["#EF9A9A", "#CE93D8", "#90CAF9"]
colors_after = ["#F44336", "#9C27B0", "#2196F3"]

for idx, fset in enumerate(["basic", "workflow"]):
    ax = axes[idx]
    x = np.arange(len(bar_metrics))
    width = 0.35

    before = [
        enriched_results[fset]["mean_noise_ratio"],
        enriched_results[fset]["negative_silence_rate"],
        enriched_results[fset]["mean_recall"],
    ]
    after = [
        llm_results[fset]["mean_noise_ratio"],
        llm_results[fset]["negative_silence_rate"],
        llm_results[fset]["mean_recall"],
    ]

    ax.bar(x - width / 2, before, width, label="Embedding only",
           color=colors_before, edgecolor="black", alpha=0.7)
    ax.bar(x + width / 2, after, width, label="+ LLM reranker",
           color=colors_after, edgecolor="black", alpha=0.9)

    # Annotate deltas
    for i, (b, a) in enumerate(zip(before, after)):
        delta = a - b
        sign = "+" if delta >= 0 else ""
        good = (delta < 0 and bar_metrics[i] == "noise") or \
               (delta > 0 and bar_metrics[i] != "noise")
        ax.annotate(f"{sign}{delta:.1%}", xy=(x[i] + width / 2, a + 0.02),
                    ha="center", va="bottom", fontsize=9, fontweight="bold",
                    color="green" if good else "red")

    ax.set_xticks(x)
    ax.set_xticklabels(bar_labels)
    ax.set_title(f"{fset.capitalize()} ({enriched_results[fset]['fixture_count']} fixtures)")
    ax.set_ylim(0, 1.15)
    ax.legend(loc="upper right")

fig.suptitle("LLM Reranker Impact (enriched + Qwen3.5-35B reasoning)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Per-tier breakdown: LLM reranker vs embedding-only (basic fixtures)
print("Per-Tier: Embedding-only vs +LLM Reranker (basic fixtures)")
print(f"{'Tier':<10} {'Emb Recall':>12} {'LLM Recall':>12} {'Delta':>8} {'Emb Noise':>12} {'LLM Noise':>12} {'Delta':>8}")
print(f"{'-'*10} {'-'*12} {'-'*12} {'-'*8} {'-'*12} {'-'*12} {'-'*8}")

emb_tiers = {t["tier"]: t for t in enriched_results["basic"]["per_tier"]}
llm_tiers = {t["tier"]: t for t in llm_results["basic"]["per_tier"]}

for tier in ["easy", "medium", "hard", "negative"]:
    et = emb_tiers.get(tier, {})
    lt = llm_tiers.get(tier, {})
    er = et.get("mean_recall", 0)
    lr = lt.get("mean_recall", 0)
    en = et.get("mean_noise_ratio", 0)
    ln = lt.get("mean_noise_ratio", 0)
    print(f"{tier:<10} {er:>12.1%} {lr:>12.1%} {lr - er:>+8.1%} {en:>12.1%} {ln:>12.1%} {ln - en:>+8.1%}")

# Also show silence for negative tier
neg_emb = emb_tiers.get("negative", {})
neg_llm = llm_tiers.get("negative", {})
print(f"\nNegative silence: embedding={neg_emb.get('silence_rate', 0):.1%} → LLM={neg_llm.get('silence_rate', 0):.1%}")
print(f"Avg retrieved: embedding={enriched_results['basic']['mean_retrieved_count']:.1f} → LLM={llm_results['basic']['mean_retrieved_count']:.2f}")

## Step 14: Full Pipeline Quality Stack

Summary table showing each stage's cumulative contribution:
**Raw bge-small → +Enrichment (v3) → +jina-code model → +LLM reranker**

Each row adds one more stage. This is the story of how we went from 84% noise to 21%.

In [ ]:
# Load v3 enriched results for jina-code
with open(RESULTS_DIR / "enriched-v3-embedding-2026-04-02.json") as f:
    v3_results = json.load(f)

# Build the pipeline stack for basic fixtures
stack = [
    ("Raw bge-small", {
        "recall": raw_results["basic"]["recall"],
        "noise": raw_results["basic"]["noise"],
        "neg_silence": raw_results["basic"]["neg_silence"],
        "precision": raw_results["basic"]["precision"],
        "p50_ms": raw_results["basic"]["p50_ms"],
    }),
    ("+Enrichment (v3)", {
        "recall": enriched_results["basic"]["mean_recall"],
        "noise": enriched_results["basic"]["mean_noise_ratio"],
        "neg_silence": enriched_results["basic"]["negative_silence_rate"],
        "precision": enriched_results["basic"]["mean_precision"],
        "p50_ms": enriched_results["basic"]["latency_p50_ms"],
    }),
    ("+jina-code model", {
        "recall": v3_results["jina_code"]["basic"]["mean_recall"],
        "noise": v3_results["jina_code"]["basic"]["mean_noise_ratio"],
        "neg_silence": v3_results["jina_code"]["basic"]["negative_silence_rate"],
        "precision": v3_results["jina_code"]["basic"]["mean_precision"],
        "p50_ms": v3_results["jina_code"]["basic"]["latency_p50_ms"],
    }),
    ("+LLM reranker", {
        "recall": llm_results["basic"]["mean_recall"],
        "noise": llm_results["basic"]["mean_noise_ratio"],
        "neg_silence": llm_results["basic"]["negative_silence_rate"],
        "precision": llm_results["basic"]["mean_precision"],
        "p50_ms": llm_results["basic"]["latency_p50_ms"],
    }),
]

print("Full Pipeline Quality Stack (basic, 354 fixtures)")
print(f"{'Stage':<22} {'Recall':>8} {'Noise':>8} {'NegSil':>8} {'Prec':>8} {'p50ms':>8}")
print(f"{'-'*22} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
for name, m in stack:
    print(f"{name:<22} {m['recall']:>8.1%} {m['noise']:>8.1%} {m['neg_silence']:>8.1%} {m['precision']:>8.1%} {m['p50_ms']:>7.0f}ms")

# Per-tier table for the best config (LLM reranker)
print(f"\n\nPer-Tier at Each Stage (basic, recall)")
print(f"{'Stage':<22} {'Easy':>8} {'Medium':>8} {'Hard':>8}")
print(f"{'-'*22} {'-'*8} {'-'*8} {'-'*8}")

# Raw bge-small per-tier from model_comp
for name, tier_src in [
    ("Raw bge-small", raw_bge["basic"]["per_tier"]),
    ("+Enrichment (v3)", enriched_results["basic"]["per_tier"]),
    ("+jina-code model", v3_results["jina_code"]["basic"]["per_tier"]),
    ("+LLM reranker", llm_results["basic"]["per_tier"]),
]:
    td = {t["tier"]: t for t in tier_src}
    print(f"{name:<22} {td['easy']['mean_recall']:>8.1%} {td['medium']['mean_recall']:>8.1%} {td['hard']['mean_recall']:>8.1%}")

In [ ]:
# Visualization: stacked pipeline progression
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

stage_names = [s[0] for s in stack]
x = np.arange(len(stage_names))

# Left: Noise reduction journey
noises = [s[1]["noise"] for s in stack]
recalls = [s[1]["recall"] for s in stack]
neg_sil = [s[1]["neg_silence"] for s in stack]

ax1.plot(x, noises, "o-", color="#F44336", linewidth=2, markersize=8, label="Noise")
ax1.plot(x, recalls, "s-", color="#4CAF50", linewidth=2, markersize=8, label="Recall")
ax1.plot(x, neg_sil, "^-", color="#9C27B0", linewidth=2, markersize=8, label="Neg Silence")

for i in range(len(stage_names)):
    ax1.annotate(f"{noises[i]:.0%}", (x[i], noises[i] + 0.03),
                 ha="center", fontsize=8, color="#F44336")
    ax1.annotate(f"{recalls[i]:.0%}", (x[i], recalls[i] - 0.05),
                 ha="center", fontsize=8, color="#4CAF50")

ax1.set_xticks(x)
ax1.set_xticklabels(stage_names, rotation=15, ha="right", fontsize=9)
ax1.set_ylabel("Score")
ax1.set_title("Pipeline Stage Progression (basic)")
ax1.set_ylim(-0.05, 1.1)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Right: Latency cost
latencies = [s[1]["p50_ms"] for s in stack]
colors = ["#4CAF50", "#4CAF50", "#FF9800", "#F44336"]
ax2.bar(x, latencies, color=colors, edgecolor="black", alpha=0.8)
for i, lat in enumerate(latencies):
    ax2.annotate(f"{lat:.0f}ms", (x[i], lat + max(latencies) * 0.02),
                 ha="center", fontsize=10, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(stage_names, rotation=15, ha="right", fontsize=9)
ax2.set_ylabel("p50 Latency (ms)")
ax2.set_title("Latency Cost per Stage")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: Enrichment is free (~1ms extra). jina-code adds ~6ms.")
print("The LLM reranker is the big cost (~1.6s) but the big win (noise 84% → 21%, silence 0% → 91%).")